In [1]:
import torch

In [8]:
def sample_context(batch_size):
  return torch.randn(batch_size,2)

In [9]:
def get_reward(context,action):
  result = (action == torch.where(context[:,0] > context[:,1],0,1))
  return result.float()

In [10]:
from torch import nn

In [11]:
class PolicyNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(2,2),
        nn.ReLU(),
        nn.Linear(2,2)
    )
  def forward(self,x):
    return self.net(x)

In [12]:
def train_step(policy,optimizer,contexts):
  logits = policy(contexts)
  dist = torch.distributions.Categorical(logits=logits)
  actions = dist.sample()
  log_probs = dist.log_prob(actions)
  rewards = get_reward(contexts,actions)
  reinforce_loss = (-log_probs * rewards).mean()
  optimizer.zero_grad()
  reinforce_loss.backward()
  optimizer.step()
  return reinforce_loss.item(),rewards.mean().item()

In [13]:
policy = PolicyNet()
optimizer = torch.optim.Adam(policy.parameters(),lr = 1e-2)
for step in range(300):
  contexts = sample_context(32)
  loss,reward = train_step(policy,optimizer,contexts)
  if (step + 1) % 50 == 0:
    print(f"loss是{loss},奖励是{reward}")

loss是0.19593720138072968,奖励是0.59375
loss是0.047660864889621735,奖励是0.8125
loss是0.026443393900990486,奖励是0.78125
loss是0.02331831492483616,奖励是0.875
loss是0.02230726182460785,奖励是0.90625
loss是0.02789481356739998,奖励是1.0


In [15]:
eval_contexts = sample_context(1000)

with torch.no_grad():
  logits = policy(eval_contexts)
  greedy_actions = torch.argmax(logits,dim=-1)
  eval_reward = get_reward(eval_contexts,greedy_actions)
accuracy = eval_reward.mean().item()
print(f"贪心策略的准确率 (Greedy Accuracy): {accuracy * 100:.2f}%")

贪心策略的准确率 (Greedy Accuracy): 99.30%
